In [3]:
import pandas as pd

def clean_table():
    """
    1. Join "First Name" and "Last Name" of Table 1 as "Name"
    2. Match with Table 2 using both "Name" and "State" columns
    3. Remove matching rows from Table 2 and save unique rows as Table 3
    """
    
    # Read the subscription table (Table 1)
    print("Reading table_subscribed.csv...")
    try:
        # Try with tab separator first, then fall back to comma
        try:
            subscribed_df = pd.read_csv('table_subscribed.csv', sep='\t')
        except:
            subscribed_df = pd.read_csv('Jan20-2026_Subscribed-Clean.csv') # exact name here
    except FileNotFoundError:
        print("Error: table_subscribed.csv not found in current directory")
        return
    except Exception as e:
        print(f"Error reading table_subscribed.csv: {e}")
        return
    
    # Read the bounce table (Table 2)
    print("Reading table_bounced.csv...")
    try:
        # Try with tab separator first, then fall back to comma
        try:
            bounced_df = pd.read_csv('table_bounced.csv', sep='\t')
        except:
            bounced_df = pd.read_csv('Feb5-2026_Comprehensive.csv') # exact name here
    except FileNotFoundError:
        print("Error: table_bounced.csv not found in current directory")
        return
    except Exception as e:
        print(f"Error reading table_bounced.csv: {e}")
        return
    
    # Display initial counts
    print(f"\nInitial count in subscribed table (Table 1): {len(subscribed_df)}")
    print(f"Initial count in bounced table (Table 2): {len(bounced_df)}")
    
    # Check column names
    print(f"\nColumns in subscribed table: {list(subscribed_df.columns)}")
    print(f"Columns in bounced table: {list(bounced_df.columns)}")
    
    # Step 1: Create "Name" column in Table 1 by joining "First Name" and "Last Name"
    print("\nStep 1: Creating 'Name' column in Table 1...")
    
    # Find First Name and Last Name columns in Table 1
    first_name_col = None
    last_name_col = None
    for col in subscribed_df.columns:
        col_lower = col.lower()
        if 'first' in col_lower and 'name' in col_lower:
            first_name_col = col
        elif 'last' in col_lower and 'name' in col_lower:
            last_name_col = col
    
    if not first_name_col or not last_name_col:
        print("Error: Could not find 'First Name' and/or 'Last Name' columns in Table 1")
        print(f"Available columns: {list(subscribed_df.columns)}")
        return
    
    print(f"Using '{first_name_col}' as First Name column")
    print(f"Using '{last_name_col}' as Last Name column")
    
    # Create Name column by combining First Name and Last Name with a space
    subscribed_df['Name'] = subscribed_df[first_name_col].fillna('') + ' ' + subscribed_df[last_name_col].fillna('')
    # Remove any leading/trailing spaces that might result from missing values
    subscribed_df['Name'] = subscribed_df['Name'].str.strip()
    
    # Step 2: Find Name and State columns in both tables
    print("\nStep 2: Finding 'Name' and 'State' columns...")
    
    # Find Name column in Table 2
    name_col_bounced = None
    for col in bounced_df.columns:
        col_lower = col.lower()
        if 'name' in col_lower and 'school' not in col_lower and 'number' not in col_lower:
            name_col_bounced = col
            break
    
    if not name_col_bounced:
        print("Error: Could not find 'Name' column in Table 2")
        print(f"Available columns: {list(bounced_df.columns)}")
        return
    
    print(f"Using '{name_col_bounced}' as Name column in Table 2")
    
    # Find State column in Table 1
    state_col_subscribed = None
    for col in subscribed_df.columns:
        if 'state' in col.lower():
            state_col_subscribed = col
            break
    
    if not state_col_subscribed:
        print("Error: Could not find 'State' column in Table 1")
        print(f"Available columns: {list(subscribed_df.columns)}")
        return
    
    print(f"Using '{state_col_subscribed}' as State column in Table 1")
    
    # Find State column in Table 2
    state_col_bounced = None
    for col in bounced_df.columns:
        if 'state' in col.lower():
            state_col_bounced = col
            break
    
    if not state_col_bounced:
        print("Error: Could not find 'State' column in Table 2")
        print(f"Available columns: {list(bounced_df.columns)}")
        return
    
    print(f"Using '{state_col_bounced}' as State column in Table 2")
    
    # Clean data for matching
    print("\nStep 3: Cleaning data for matching...")
    
    # Clean names and states in Table 1
    subscribed_df['Name_Clean'] = subscribed_df['Name'].str.strip().str.lower()
    subscribed_df['State_Clean'] = subscribed_df[state_col_subscribed].astype(str).str.strip().str.upper()
    
    # Clean names and states in Table 2
    bounced_df['Name_Clean'] = bounced_df[name_col_bounced].astype(str).str.strip().str.lower()
    bounced_df['State_Clean'] = bounced_df[state_col_bounced].astype(str).str.strip().str.upper()
    
    # Create combined key columns for matching
    subscribed_df['Match_Key'] = subscribed_df['Name_Clean'] + '||' + subscribed_df['State_Clean']
    bounced_df['Match_Key'] = bounced_df['Name_Clean'] + '||' + bounced_df['State_Clean']
    
    # Get unique match keys from Table 1
    subscribed_match_keys = set(subscribed_df['Match_Key'].dropna().tolist())
    
    # Step 4: Filter Table 2 to remove rows with matching Name AND State
    print("\nStep 4: Filtering Table 2 to remove rows with matching Name AND State...")
    
    # Create a mask for rows in Table 2 that are NOT in Table 1 match keys
    mask = ~bounced_df['Match_Key'].isin(subscribed_match_keys)
    table3_df = bounced_df[mask].copy()
    
    # Remove the temporary columns we added
    table3_df = table3_df.drop(['Name_Clean', 'State_Clean', 'Match_Key'], axis=1)
    
    # Display results
    removed_count = len(bounced_df) - len(table3_df)
    print(f"Removed {removed_count} rows with matching Name AND State")
    print(f"Final count in Table 3: {len(table3_df)}")
    
    # Show some examples of matches
    matches_df = bounced_df[~mask]
    if not matches_df.empty:
        print(f"\nExamples of matches found (Name + State):")
        for i, row in matches_df.head(5).iterrows():
            print(f"  - {row[name_col_bounced]} ({row[state_col_bounced]})")
        if len(matches_df) > 5:
            print(f"  ... and {len(matches_df) - 5} more")
    
    # Save the filtered Table 3
    print("\nSaving filtered table to table_clean.csv...")
    try:
        table3_df.to_csv('table_clean.csv', index=False)
        print("Successfully saved table_clean.csv")
        
        # Show examples of kept rows
        if not table3_df.empty:
            print(f"\nExamples of kept rows in Table 3:")
            for i, row in table3_df.head(5).iterrows():
                print(f"  {i+1}. {row[name_col_bounced]} - {row[state_col_bounced]}")
            if len(table3_df) > 5:
                print(f"  ... and {len(table3_df) - 5} more")
    except Exception as e:
        print(f"Error saving table_clean.csv: {e}")

if __name__ == "__main__":
    clean_table()

Reading table_subscribed.csv...
Reading table_bounced.csv...

Initial count in subscribed table (Table 1): 1292
Initial count in bounced table (Table 2): 1033

Columns in subscribed table: ['Email Address', 'First Name', 'Last Name', 'Address', 'Phone Number', 'Birthday', 'Company', 'Role', 'District', 'State', 'MEMBER_RATING', 'OPTIN_TIME', 'OPTIN_IP', 'CONFIRM_TIME', 'CONFIRM_IP', 'GMTOFF', 'DSTOFF', 'TIMEZONE', 'CC', 'REGION', 'LAST_CHANGED', 'LEID', 'EUID', 'NOTES', 'TAGS']
Columns in bounced table: ['Name', 'Email', 'School Number', 'School Name', 'Type', 'City', 'State']

Step 1: Creating 'Name' column in Table 1...
Using 'First Name' as First Name column
Using 'Last Name' as Last Name column

Step 2: Finding 'Name' and 'State' columns...
Using 'Name' as Name column in Table 2
Using 'State' as State column in Table 1
Using 'State' as State column in Table 2

Step 3: Cleaning data for matching...

Step 4: Filtering Table 2 to remove rows with matching Name AND State...
Removed 5 r